> **Siddu Gundeboina (25101571) - TRANSFORM A: DATA QUALITY & CLEANING**: this notebook has only my section. In the final `01_ETL_Pipeline.ipynb`, it is pasted in order after the earlier sections, and everything runs top to bottom.

# PerfectPetClub – Customer Churn ETL Pipeline

**Module:** Data Engineering Team Presentation &nbsp;|&nbsp; **Notebook 1 of 2:** ETL pipeline

**Goal:** build a clean, one-row-per-customer dataset for analysing **customer churn**, where churn means a customer has **cancelled the auto-renew** of their subscription (`subscription.auto_renew == 0`).

| Section | Owner | What it does |
|---|---|---|
| 0. Setup & configuration | Team | Imports, paths, logging |
| 1. Extract | Bhavana Padala | Read SQLite DB, CRM CSV, server log CSV, survey CSV + question metadata |
| 2. Transform A – data quality & cleaning | Siddu Gundeboina | Data quality checks, type conversion, parsing, cleaning |
| 3. Transform B – feature engineering | Pravalika | Build customer-level features and the churn target |
| 4. Load | Pravalika | Write and verify the Parquet files |
| 5. Testing & validation | Denya Sree Adapa | Unit tests and end-to-end checks on the output |
| Repo, README, merge, HTML export | Ramya Sree Kasula & Denya Sree Adapa | Git repository, merging branches, run-all + HTML export |

**Architecture**

```
 SQLite DB ─┐
 CRM CSV  ──┤   extract_*()  ──►  raw dict  ──►  run_quality_checks() ──► clean_*()  ──►  build_*_features()
 Server log ┤                                   (DQ report)               (typed,         (one row per customer)
 Survey CSV ┘                                                              parsed)               │
                                                                                                  ▼
                                                               load_parquet()  ──►  output/customer_churn.parquet
                                                                                    output/customer_churn_model_ready.parquet
```

## 0. Setup & configuration

In [ ]:
import ast
import logging
import re
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

# ---- Configuration (change paths here, not inside functions) ----
DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

SOURCES = {
    "db": DATA_DIR / "PerfectPetClub.db",
    "crm": DATA_DIR / "PerfectPetClub_CRM_report_1.csv",
    "server_log": DATA_DIR / "PerfectPetClub_server_log_1.csv",
    "survey": DATA_DIR / "PerfectPetClub_survey.csv",
    "survey_questions": DATA_DIR / "PerfectPetClub_survey_questions.txt",
}
DB_TABLES = ["customer", "subscription", "subscription_type", "sale", "sale_product",
             "product", "product_category", "payment", "product_review"]

OUTPUT_FILE = OUTPUT_DIR / "customer_churn.parquet"
MODEL_READY_FILE = OUTPUT_DIR / "customer_churn_model_ready.parquet"

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", force=True)
log = logging.getLogger("ppc_etl")
log.info("pandas %s | numpy %s", pd.__version__, np.__version__)

## 2. Transform A: data quality checks & cleaning  *(owner: Siddu Gundeboina)*

**Approach:** run all checks *before* changing anything, so the report describes the raw data. Each check gives:
`check`, `table`, `failed_rows`, `severity` (ERROR stops the pipeline; WARN is logged and carried forward).

Categories of check: **completeness** (nulls), **uniqueness** (primary keys / duplicate rows), **validity** (ranges, formats),
**consistency** (dates in the right order, payment = basket total), **referential integrity** (foreign keys and cross-source joins).

In [ ]:
EMAIL_RE = r"^[\w.+\-']+@[\w\-]+(\.[\w\-]+)+$"


def _check(results, name, table, failed, severity="ERROR", note=""):
    results.append({"check": name, "table": table, "failed_rows": int(failed),
                    "status": "PASS" if failed == 0 else severity, "note": note})


def run_quality_checks(raw: dict) -> pd.DataFrame:
    """Run data-quality checks on the RAW data and return a report DataFrame."""
    r = []
    cust, sub, sale = raw["customer"], raw["subscription"], raw["sale"]

    # --- Completeness ---
    for t in ["customer", "subscription", "crm", "server_log", "survey"]:
        _check(r, "no nulls", t, raw[t].isna().sum().sum(), "WARN")
    # Missing delivery dates are expected for subscription-only orders, so check *unexpected* ones
    sub_product_ids = raw["product"].loc[raw["product"]["name"].str.contains("subscription", case=False), "id"]
    sub_only_sales = raw["sale_product"].groupby("sale_id")["product_id"].apply(lambda s: s.isin(sub_product_ids).all())
    unexpected_missing = sale["delivered_date"].isna() & ~sale["id"].map(sub_only_sales).fillna(False)
    _check(r, "missing delivery only on subscription orders", "sale", unexpected_missing.sum())

    # --- Uniqueness ---
    for t in DB_TABLES:
        _check(r, "unique primary key", t, raw[t]["id"].duplicated().sum())
    for t in ["crm", "server_log", "survey"]:
        _check(r, "no duplicate rows", t, raw[t].duplicated().sum(), "WARN")
    _check(r, "one survey per customer", "survey", raw["survey"]["email"].str.lower().duplicated().sum(), "WARN")
    _check(r, "unique customer email", "customer", cust["email"].str.lower().duplicated().sum())

    # --- Validity ---
    _check(r, "valid email format", "customer", (~cust["email"].str.match(EMAIL_RE)).sum())
    _check(r, "auto_renew in {0,1}", "subscription", (~sub["auto_renew"].isin([0, 1])).sum())
    _check(r, "star_rating 1-5", "product_review", (~raw["product_review"]["star_rating"].between(1, 5)).sum())
    _check(r, "quantity > 0", "sale_product", (raw["sale_product"]["quantity"] <= 0).sum())
    _check(r, "price > 0", "product", (raw["product"]["price"] <= 0).sum())
    _check(r, "amount > 0", "payment", (raw["payment"]["amount"] <= 0).sum())
    for c in ["nps_recommend", "satisfaction_score", "online_comparison_freq"]:
        _check(r, f"{c} 0-10", "survey", (~raw["survey"][c].between(0, 10)).sum())
    _check(r, "income band parseable", "survey",
           (~raw["survey"]["reported_income"].str.match(r"^\d+k-\d+k\+?$")).sum())
    _check(r, "CRM type in {complaint, query}", "crm", (~raw["crm"]["type"].isin(["complaint", "query"])).sum())

    # --- Consistency ---
    sd, ed = pd.to_datetime(sub["start_date"]), pd.to_datetime(sub["end_date"])
    _check(r, "end_date >= start_date", "subscription", (ed < sd).sum())
    od, dd = pd.to_datetime(sale["order_date"]), pd.to_datetime(sale["delivered_date"])
    _check(r, "delivered >= ordered", "sale", (dd < od).sum())
    basket = (raw["sale_product"].merge(raw["product"], left_on="product_id", right_on="id")
              .assign(line=lambda d: d["quantity"] * d["price"]).groupby("sale_id")["line"].sum())
    paid = sale.set_index("id")["payment_id"].map(raw["payment"].set_index("id")["amount"])
    _check(r, "payment amount = basket total", "sale/payment", (basket.reindex(paid.index) != paid).sum())

    # --- Referential integrity ---
    emails = set(cust["email"].str.lower())
    _check(r, "FK customer.subscription_id", "customer", (~cust["subscription_id"].isin(sub["id"])).sum())
    _check(r, "FK sale.customer_id", "sale", (~sale["customer_id"].isin(cust["id"])).sum())
    _check(r, "FK sale_product.product_id", "sale_product", (~raw["sale_product"]["product_id"].isin(raw["product"]["id"])).sum())
    _check(r, "CRM customer_id exists", "crm", (~raw["crm"]["customer_id"].isin(cust["id"])).sum())
    _check(r, "server_log login matches a customer", "server_log", (~raw["server_log"]["login"].str.lower().isin(emails)).sum())
    _check(r, "survey email matches a customer", "survey", (~raw["survey"]["email"].str.lower().isin(emails)).sum())

    # --- Metadata check: are the survey questions distinct? ---
    q = raw["survey_questions"]
    texts = {k: re.sub(r"^Q\d+\.\s*", "", v.split("[")[0]).strip() for k, v in q.items() if k != "email"}
    dup_q = pd.Series(texts).duplicated(keep=False)
    _check(r, "survey questions have distinct wording", "survey_questions", dup_q.sum(), "WARN",
           note="duplicate wording: " + ", ".join(dup_q[dup_q].index) if dup_q.any() else "")

    report = pd.DataFrame(r)
    log.info("Data quality: %d checks | %d PASS | %d WARN | %d ERROR", len(report),
             (report.status == "PASS").sum(), (report.status == "WARN").sum(), (report.status == "ERROR").sum())
    return report


dq_report = run_quality_checks(raw)
dq_report.style.apply(
    lambda row: ["background-color:#fde2e1" if row.status == "ERROR"
                 else "background-color:#fff4d6" if row.status == "WARN" else ""] * len(row), axis=1)

In [ ]:
# Stop the pipeline if any ERROR-level check fails, rather than writing a bad dataset
errors = dq_report[dq_report["status"] == "ERROR"]
if not errors.empty:
    raise ValueError(f"Data quality ERRORs found - pipeline stopped:\n{errors}")
print("No blocking data quality errors.")
print(dq_report[dq_report["status"] == "WARN"][["check", "table", "failed_rows", "note"]].to_string(index=False))

**Finding:** the survey-questions file shows that **Q3 (`satisfaction_score`) uses exactly the same wording as Q2 (`nps_recommend`)**:
*"Would you recommend us to a friend?"*. So `satisfaction_score` may not measure satisfaction at all. We keep the column but
mark it as low-reliability in the dashboard and recommend that the business fixes the survey question.

### Cleaning functions
Small single-purpose parsers (easy to unit-test), then one `clean_*` function for each source.

In [ ]:
def parse_platform(value) -> str:
    """"['e-mail']" -> 'e-mail'. Falls back to stripping brackets if the value is not a valid list literal."""
    try:
        parsed = ast.literal_eval(value)
        return str(parsed[0]) if isinstance(parsed, (list, tuple)) and parsed else str(parsed)
    except (ValueError, SyntaxError):
        return str(value).strip("[]'\" ")


UA_RE = re.compile(r"\((?P<device_type>[^;]+);\s*(?P<device_model>[^)]+)\)")

def parse_user_agent(ua: str) -> tuple[str, str]:
    """Extract (device_type, device_model) from a truncated user-agent string."""
    m = UA_RE.search(str(ua))
    return (m["device_type"].strip(), m["device_model"].strip()) if m else ("Unknown", "Unknown")


def parse_income(band: str) -> float:
    """'26k-30k' -> 28000.0 (band midpoint); '198k-200k+' -> 199000.0; invalid -> NaN."""
    m = re.match(r"^(\d+)k-(\d+)k\+?$", str(band).strip())
    if not m:
        return np.nan
    low, high = int(m[1]), int(m[2])
    return (low + high) / 2 * 1000


def _to_dt(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, errors="coerce")   # bad dates become NaT instead of crashing


def clean_db(raw: dict) -> dict:
    """Type-convert and tidy the relational tables. Prices/amounts are stored in pence -> convert to GBP."""
    c = {}
    c["customer"] = (raw["customer"]
                     .drop(columns=["name", "password_hash", "address_line_1", "address_line_2", "postcode"])  # PII minimisation
                     .rename(columns={"id": "customer_id"})
                     .assign(email=lambda d: d["email"].str.strip().str.lower(),
                             county=lambda d: d["county"].str.strip().str.title()))
    c["subscription"] = (raw["subscription"]
                         .rename(columns={"id": "subscription_id"})
                         .assign(start_date=lambda d: _to_dt(d["start_date"]),
                                 end_date=lambda d: _to_dt(d["end_date"]),
                                 auto_renew=lambda d: d["auto_renew"].astype(int))
                         .merge(raw["subscription_type"].rename(columns={"id": "subscription_type_id",
                                                                         "name": "subscription_type"}),
                                on="subscription_type_id", how="left"))
    c["sale"] = (raw["sale"].rename(columns={"id": "sale_id"})
                 .assign(order_date=lambda d: _to_dt(d["order_date"]),
                         expected_date=lambda d: _to_dt(d["expected_date"]),
                         delivered_date=lambda d: _to_dt(d["delivered_date"]))
                 .merge(raw["payment"].rename(columns={"id": "payment_id"}), on="payment_id", how="left")
                 .assign(amount_gbp=lambda d: d["amount"] / 100))
    c["sale_product"] = (raw["sale_product"]
                         .merge(raw["product"].rename(columns={"id": "product_id", "name": "product_name"}),
                                on="product_id", how="left")
                         .merge(raw["product_category"].rename(columns={"id": "product_category_id",
                                                                        "name": "category"}),
                                on="product_category_id", how="left")
                         .assign(price_gbp=lambda d: d["price"] / 100,
                                 is_subscription_item=lambda d: d["product_name"].str.contains("subscription", case=False),
                                 category=lambda d: d["category"].fillna("Subscription")))
    c["product_review"] = raw["product_review"].copy()
    return c


def clean_crm(crm: pd.DataFrame) -> pd.DataFrame:
    return (crm.rename(columns={"recieved_datatime": "received_datetime"})      # fix source typo
               .assign(received_datetime=lambda d: _to_dt(d["received_datetime"]),
                       platform=lambda d: d["platform"].map(parse_platform),
                       type=lambda d: d["type"].str.strip().str.lower())
               .drop_duplicates())


def clean_server_log(logs: pd.DataFrame) -> pd.DataFrame:
    ua = logs["user_agent"].map(parse_user_agent)
    return (logs.assign(datetime=_to_dt(logs["datetime"]),
                        login=logs["login"].str.strip().str.lower(),
                        device_type=ua.str[0], device_model=ua.str[1])
                .drop(columns=["user_agent"])
                .drop_duplicates())


def clean_survey(survey: pd.DataFrame) -> pd.DataFrame:
    score_cols = ["nps_recommend", "satisfaction_score", "online_comparison_freq"]
    out = survey.assign(email=survey["email"].str.strip().str.lower(),
                        income_gbp=survey["reported_income"].map(parse_income))
    out[score_cols] = out[score_cols].apply(pd.to_numeric, errors="coerce").clip(0, 10)
    return out.drop_duplicates(subset="email", keep="last")      # keep the latest answer if a customer submits twice


def clean_all(raw: dict) -> dict:
    clean = clean_db(raw)
    clean["crm"] = clean_crm(raw["crm"])
    clean["server_log"] = clean_server_log(raw["server_log"])
    clean["survey"] = clean_survey(raw["survey"])
    for k, v in clean.items():
        log.info("Cleaned %-15s %6d rows", k, len(v))
    return clean


clean = clean_all(raw)
display(clean["crm"].head(3), clean["server_log"].head(3), clean["survey"].head(3))